# TADC-SBM: a Time-varying, Attributed, Degree-Corrected Stochastic Block Model

[![PyPI](https://badge.fury.io/py/tadc-sbm.svg)](https://badge.fury.io/py/tadc-sbm)
[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nelsonaloysio/tadc-sbm/blob/main/notebook.ipynb)
[![DOI](https://img.shields.io/badge/doi-10.1109/ISCC65549.2025.11326334-blue)](https://doi.org/10.1109/ISCC65549.2025.11326334)
[![PDF](https://img.shields.io/badge/pdf-Paper-red)](https://nelsonaloysio.github.io/files/tadcsbm2025.pdf)
[![License](https://img.shields.io/pypi/l/tadc-sbm)](https://github.com/nelsonaloysio/tadc-sbm/blob/main/LICENSE.md)

In [ ]:
# required for Google Colab (1/2)
!pip install -q condacolab
import condacolab
# condacolab.install()
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

In [ ]:
# required for Google Colab (2/2)
!conda install -y -q -c conda-forge graph-tool
!pip install -q tadc-sbm

___

In [ ]:
from collections import Counter
import networkx_temporal as tx
import numpy as np
from tadcsbm import (
    tadcsbm_simulator,
    generate_block_matrix,
    generate_transition_matrix,
    generate_community_vector,
    gt_to_nx_temporal,
    inspect_sbm
)

## Overview

This example generates the TADC-SBM graph sequence used in the paper, where nodes are assigned to communities and edges are generated according to the degree-corrected stochastic block model (DC-SBM) with time-varying community structure and fixed attributes. The generated graph sequence is available as a [graph-tool](https://graph-tool.skewed.de/) graph object, and node features are stored as NumPy arrays. The network is also saved to disk in a compressed format with the [NetworkX-Temporal](https://networkx-temporal.org) library.

In [ ]:
output_dir = "output"  # Directory to save output files (graphs and features).
output_ext = "gexf"    # Format to save graph files (e.g., "gexf", "graphml", etc.).

Main parameters for the TADC-SBM simulation are defined below. Modify these to generate different types of graphs; default parameters are set to match the graph sequence used in the paper.

In [ ]:
k = 8        # Number of communities.
t = 8        # Number of snapshots.
n = 1024     # Number of vertices.
e = 10240    # Number of edges (expected - consider division by 2 as graphs are undirected).
d = 20       # Expected average node degree (before degree-correction and edge sampling).
d_inter = 2  # Expected average inter-community degree (used to compute p and q for block matrix).
eta = 0.75   # Community stability rate, where 1 means nodes never change communities.
gamma = 0    # If 0, node transition probabilities depend on current community affiliation.
beta = 1.0   # Edge sampling rate in [0, 1], where 1 means all edges are included in the graph.

The generated network may be used for benchmarking dynamic graph learning algorithms, such as node classification, link prediction, and community detection, under controlled experimentation settings. Nodes and edges may both be (optionally) assigned attributes with different signal-to-noise ratios and hierarchical (nested, overlapping) community structures.

In [ ]:
feature_dim = 32                 # Dimension of node features (optional).
feature_center_distance = 6.0    # Distance between centroids.
feature_cluster_variance = 1.0   # Variance of feature clusters.
edge_feature_dim = 0             # Dimension of edge features (optional).
edge_center_distance = 6.0       # Distance between edge feature centroids.
edge_cluster_variance = 1.0      # Variance of edge feature clusters.
reverse_snapshot_order = True    # If True, snapshot indices are inverted.
uniform_all = False              # If False, do not consider self-transitions.
random_seed = None               # Random seed for reproducibility (optional).

Generate block matrix $\mathbf{B}$, transition probability matrix $\mathbf{\tau}$, and initial community assignments $\mathbf{Z}$. Here we set $p=(d-d^\star)/n, q=d^\star/n$, where $d$ is the expected average node degree and $d^\star$ is the expected average inter-community degree -- meaning that, if each node were to have on average 20 edges, 2 of them would be expected to other communities (end results vary). This results in a strong community structure, $p/q = (d-d^\star)/d^\star$, while $\eta$ varies in $[1, 0.75, 0.5, 0.25, 0]$ in the paper evaluation benchmarks.

In [ ]:
p = (d - d_inter) / n  # Within-community edge probability.
q = d_inter / n        # Between-community edge probability.
B = generate_block_matrix(k, p=p, q=q)
B

In [ ]:
tau = generate_transition_matrix(k, eta, uniform_all=False)
tau

In [ ]:
z = generate_community_vector(n, k, shuffle=False)
z

Generate TADC-SBM graph sequence with the specified parameters.

In [ ]:
sbm = tadcsbm_simulator(
    snapshots=t,
    num_vertices=n,
    num_edges=e,
    pi=[v/len(z) for k, v in Counter(z).items()],
    prop_mat=B,
    tau_mat=tau,
    num_feature_groups=k,
    feature_dim=feature_dim,
    feature_center_distance=feature_center_distance,
    feature_cluster_variance=feature_cluster_variance,
    edge_feature_dim=edge_feature_dim,
    edge_center_distance=edge_center_distance,
    edge_cluster_variance=edge_cluster_variance,
    reverse_snapshot_order=reverse_snapshot_order,
    fixed_probabilities=gamma,
    edge_sampling_rate=beta,
    random_seed=random_seed,
)

Print the number of vertices and edges in each snapshot, along with the community and link distribution.

In [ ]:
inspect_sbm(sbm, mat=B)

Compose graph-tool graphs as a single NetworkX-Temporal multigraph.

In [ ]:
TG = gt_to_nx_temporal(sbm.graph)
print(TG)

Save graph as compressed file (with timestamps and community labels) and node and edge features as NumPy arrays.

In [ ]:
!mkdir -p {output_dir}
print("Saving graph...")
tx.write_graph(TG, f"{output_dir}/graphs.{output_ext}.zip")
if feature_dim > 0:
    print("Saving node features...")
    np.save(f"{output_dir}/features_node.npy", sbm.node_features1)
if edge_feature_dim > 0:
    print("Saving edge features...")
    np.save(f"{output_dir}/features_edge.npy", sbm.edge_features)

___

Inspect the transition probability function $P(\mathbf{Z}^{(t+1)} \vert \mathbf{Z}^{(t)}, \mathbf{\tau})$ for a single snapshot to verify that the generated community structure in $t+1$ is consistent with the average expected transition.

In [ ]:
from tadcsbm.simulations.sbm_simulator import _TransitionNodeMemberships

old_memberships = sbm.graph_memberships[-1]
new_memberships = _TransitionNodeMemberships(old_memberships, tau)
transitioned_nodes = np.sum(new_memberships != old_memberships)

transition_diag = 1 - np.diag(tau).sum()/len(tau)
expected_nodes = int(transition_diag * len(old_memberships))
print(f"Transitioned nodes: {transitioned_nodes} (expected: {expected_nodes})")

Inspect the generated SBM by printing the number of vertices and edges in each snapshot, along with the community distribution. Similar to `inspect_sbm`, but a bit faster and less informative.

In [ ]:
V, E, T = 0, 0, 0

for i, (g, memberships) in enumerate(zip(sbm.graph, sbm.graph_memberships)):
    V += g.num_vertices()
    E += g.num_edges()
    T += (memberships != sbm.graph_memberships[i-1])[g.get_vertices()].sum() if i else 0
    print(f"Snapshot {i+1}/{len(sbm.graph)}: {g.num_vertices()} nodes, {g.num_edges()} edges, "
          f"communities: {np.bincount(memberships)}")

print(f"\nTotal nodes across snapshots: {V}"
      f"\nTotal edges across snapshots: {E}"
      f"\nTotal transitions across snapshots: {T} ({T/(V-g.num_vertices() or 1):.2%})")

Inspect if the average proportion of within-/between-community edges is consistent with the expected average degree $d$ and inter-community degree $d^\star$, i.e., $D_{\text{within}} / D_{\text{between}} \approx p / q$.

In [ ]:
ratios = []
expected_ratio = p / q if q > 0 else 1

for t, g in enumerate(TG):
    D = {}

    for edge in g.edges(data=True):
        u, v, data = edge
        c_u = g.nodes[u]["community"]
        c_v = g.nodes[v]["community"]
        D[c_u == c_v] = D.get(c_u == c_v, 0) + 1

    within = sum(v for k, v in D.items() if k == True)
    between = sum(v for k, v in D.items() if k == False)
    ratio = within / between if between > 0 else 1

    print(f"Snapshot {t}/{len(TG)}: "
          f"within-community: {within}, "
          f"between-community: {between}, "
          f"within/between edge ratio: {ratio:.2f}")

    ratios.append(ratio)

print(f"\nAverage within/between ratio across snapshots: {np.mean(ratios):.2f}",
      f"\nExpected within/between ratio (p/q) overall: {expected_ratio:.2f}")